# Phase 4 Validation (Variants / Analysis)

Manual validation notebook for `pm_spark.variants.analysis` using `create_sample_event_log()`.

**Implementation notes (post-refactor):**

- Fingerprints order events by `(timestamp, tie-break, activity)` — optional `event_order_col`, else a synthetic monotonic id (same idea as `preparation._resolve_event_order_column`).
- `variant_frequency_ranking` builds ranks on the **driver** (no global unpartitioned window); output is **unordered** — use `.orderBy("variant_rank", F.col("case_count").desc())` to display.
- `variant_coverage_ratio` recomputes cumulative share on the driver and returns an explicit schema (`df.schema` + `cumulative_coverage` as `DoubleType`).
- `filter_top_n_variants` and `flag_rare_variants` return a **cached** DataFrame (materialized inside the function). Call **`df.unpersist()`** when you no longer need that result to free memory.
- `variant_drift_over_time` uses **`dense_rank`** per period (aligned with global variant ranking) and derives case start in the same fingerprint aggregation as the trace.

Covered functions:
- `case_variant_fingerprint()`
- `variant_frequency_ranking()`
- `filter_top_n_variants()`
- `variant_coverage_ratio()`
- `variant_attribute_profile()`
- `detect_variant_loops()`
- `flag_rare_variants()`
- `variant_drift_over_time()`

In [1]:
from pathlib import Path
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

project_root = Path.cwd().resolve()
if project_root.name == "dev":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("pm_spark_phase4_validation").getOrCreate()

from dev.sample_data import create_sample_event_log
from pm_spark.variants.analysis import (
    case_variant_fingerprint,
    detect_variant_loops,
    filter_top_n_variants,
    flag_rare_variants,
    variant_attribute_profile,
    variant_coverage_ratio,
    variant_drift_over_time,
    variant_frequency_ranking,
)

def show_df(df, n=200):
    df.show(n, truncate=False)

26/03/22 12:40:59 WARN Utils: Your hostname, MacBook-Air-von-Alex.local resolves to a loopback address: 127.0.0.1; using 192.168.178.137 instead (on interface en0)
26/03/22 12:40:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 12:41:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/22 12:41:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 54947)
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/Users/alexholzer/Library/Python/3.9/lib/pyth

In [2]:
event_log = create_sample_event_log(spark)
show_df(event_log.orderBy("case_key", "timestamp", "activity"))

+--------+-------------+-------------------+--------+----------+
|case_key|activity     |timestamp          |resource|department|
+--------+-------------+-------------------+--------+----------+
|C001    |A            |2026-03-18 09:00:00|R1      |D1        |
|C001    |B            |2026-03-18 09:05:00|R1      |D1        |
|C001    |C            |2026-03-18 09:10:00|R1      |D1        |
|C001    |D            |2026-03-18 09:20:00|R1      |D1        |
|C002    |A            |2026-03-18 09:00:00|R1      |D1        |
|C002    |B            |2026-03-18 09:04:00|R1      |D1        |
|C002    |Manual_Review|2026-03-18 09:08:00|R1      |D1        |
|C002    |C            |2026-03-18 09:15:00|R2      |D2        |
|C002    |D            |2026-03-18 09:25:00|R2      |D2        |
|C003    |A            |2026-03-18 10:00:00|NULL    |D1        |
|C003    |C            |2026-03-18 10:10:00|R3      |NULL      |
|C003    |D            |2026-03-18 10:30:00|R3      |D3        |
|C004    |A            |2

In [3]:
# 4.1 case_variant_fingerprint() — optional event_order_col; default synthetic tie-break
df_fp = case_variant_fingerprint(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="variant_fingerprint",
)
show_df(df_fp.orderBy("case_key"))

# Expected: C001 -> A→B→C→D ; C006 includes loop pattern in string

+--------+---------------------+
|case_key|variant_fingerprint  |
+--------+---------------------+
|C001    |A→B→C→D              |
|C002    |A→B→Manual_Review→C→D|
|C003    |A→C→D                |
|C004    |A→B→B→C→D            |
|C005    |A→B→P→C→D            |
|C006    |A→B→A→C→D            |
|C007    |A→B→C→D              |
+--------+---------------------+



In [4]:
# 4.2 variant_frequency_ranking() — driver-side dense rank; unordered by default
df_ranked = variant_frequency_ranking(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
)
show_df(df_ranked.orderBy("variant_rank", F.col("case_count").desc()))

+---------------------+----------+------------+
|variant_fingerprint  |case_count|variant_rank|
+---------------------+----------+------------+
|A→B→C→D              |2         |1           |
|A→B→Manual_Review→C→D|1         |2           |
|A→C→D                |1         |2           |
|A→B→A→C→D            |1         |2           |
|A→B→P→C→D            |1         |2           |
|A→B→B→C→D            |1         |2           |
+---------------------+----------+------------+



In [5]:
# 4.3 filter_top_n_variants() (e.g. top 2 variant ranks)
# Return value is cached + materialized inside the library — unpersist when done.
df_top2 = filter_top_n_variants(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    n=2,
)
show_df(df_top2.orderBy("case_key", "timestamp"))
print("distinct cases in filtered log:", df_top2.select("case_key").distinct().count())
df_top2.unpersist()

+--------+-------------+-------------------+--------+----------+
|case_key|activity     |timestamp          |resource|department|
+--------+-------------+-------------------+--------+----------+
|C001    |A            |2026-03-18 09:00:00|R1      |D1        |
|C001    |B            |2026-03-18 09:05:00|R1      |D1        |
|C001    |C            |2026-03-18 09:10:00|R1      |D1        |
|C001    |D            |2026-03-18 09:20:00|R1      |D1        |
|C002    |A            |2026-03-18 09:00:00|R1      |D1        |
|C002    |B            |2026-03-18 09:04:00|R1      |D1        |
|C002    |Manual_Review|2026-03-18 09:08:00|R1      |D1        |
|C002    |C            |2026-03-18 09:15:00|R2      |D2        |
|C002    |D            |2026-03-18 09:25:00|R2      |D2        |
|C003    |A            |2026-03-18 10:00:00|NULL    |D1        |
|C003    |C            |2026-03-18 10:10:00|R3      |NULL      |
|C003    |D            |2026-03-18 10:30:00|R3      |D3        |
|C004    |A            |2

DataFrame[case_key: string, activity: string, timestamp: timestamp, resource: string, department: string]

In [6]:
# 4.4 variant_coverage_ratio() on ranking output (driver cumulative; explicit schema)
df_cov = variant_coverage_ratio(
    df=df_ranked,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    coverage_threshold=None,
)
show_df(df_cov.orderBy("variant_rank", F.col("case_count").desc()))

df_cov_80 = variant_coverage_ratio(
    df=df_ranked,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    coverage_threshold=0.8,
)
show_df(df_cov_80.orderBy("variant_rank", F.col("case_count").desc()))

+---------------------+----------+------------+-------------------+
|variant_fingerprint  |case_count|variant_rank|cumulative_coverage|
+---------------------+----------+------------+-------------------+
|A→B→C→D              |2         |1           |0.2857142857142857 |
|A→C→D                |1         |2           |1.0                |
|A→B→P→C→D            |1         |2           |0.8571428571428571 |
|A→B→Manual_Review→C→D|1         |2           |0.7142857142857143 |
|A→B→B→C→D            |1         |2           |0.5714285714285714 |
|A→B→A→C→D            |1         |2           |0.42857142857142855|
+---------------------+----------+------------+-------------------+

+---------------------+----------+------------+-------------------+
|variant_fingerprint  |case_count|variant_rank|cumulative_coverage|
+---------------------+----------+------------+-------------------+
|A→B→C→D              |2         |1           |0.2857142857142857 |
|A→B→P→C→D            |1         |2           |

In [7]:
# 4.5 variant_attribute_profile() — mock case-level attributes
rows = [
    ("C001", 25.0, True, "P1"),
    ("C002", 40.0, False, "P1"),
    ("C003", 30.0, True, "P2"),
    ("C004", 35.0, False, "P1"),
    ("C005", 20.0, True, "P2"),
    ("C006", 28.0, False, "P1"),
    ("C007", 22.0, True, "P2"),
]
attrs = spark.createDataFrame(
    rows,
    schema=T.StructType(
        [
            T.StructField("case_key", T.StringType(), False),
            T.StructField("throughput_min", T.DoubleType(), True),
            T.StructField("is_stp", T.BooleanType(), True),
            T.StructField("product_line", T.StringType(), True),
        ]
    ),
)
df_profile = variant_attribute_profile(
    df=df_fp,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    attributes_df=attrs,
    attribute_join_col="case_key",
    fingerprint_col="variant_fingerprint",
    continuous_cols=["throughput_min"],
    categorical_cols=["is_stp", "product_line"],
)
show_df(df_profile)

+---------------------+------------------+-------------------+----------------------+----------------------------+
|variant_fingerprint  |case_count_profile|throughput_min_mean|is_stp_distinct_values|product_line_distinct_values|
+---------------------+------------------+-------------------+----------------------+----------------------------+
|A→B→A→C→D            |1                 |28.0               |[false]               |[P1]                        |
|A→B→B→C→D            |1                 |35.0               |[false]               |[P1]                        |
|A→B→C→D              |2                 |23.5               |[true]                |[P1, P2]                    |
|A→C→D                |1                 |30.0               |[true]                |[P2]                        |
|A→B→Manual_Review→C→D|1                 |40.0               |[false]               |[P1]                        |
|A→B→P→C→D            |1                 |20.0               |[true]            

In [8]:
# 4.6 detect_variant_loops()
df_loops_fp = detect_variant_loops(
    df=df_fp,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    fingerprint_col="variant_fingerprint",
    min_repetitions=2,
    include_looping_activities=True,
)
show_df(df_loops_fp.orderBy("variant_fingerprint"))

+---------------------+------------------+--------+
|variant_fingerprint  |looping_activities|has_loop|
+---------------------+------------------+--------+
|A→B→A→C→D            |[A]               |true    |
|A→B→B→C→D            |[B]               |true    |
|A→B→C→D              |NULL              |false   |
|A→B→Manual_Review→C→D|NULL              |false   |
|A→B→P→C→D            |NULL              |false   |
|A→C→D                |NULL              |false   |
+---------------------+------------------+--------+



In [9]:
# 4.7 flag_rare_variants() — cached + materialized return; unpersist when done
df_rare_abs = flag_rare_variants(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    mode="absolute",
    min_count=2,
)
show_df(df_rare_abs.select("case_key", "is_rare_variant").distinct().orderBy("case_key"))
df_rare_abs.unpersist()

df_rare_rel = flag_rare_variants(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    mode="relative",
    min_count=5,
    min_share=0.15,
)
show_df(df_rare_rel.select("case_key", "is_rare_variant").distinct().orderBy("case_key"))
df_rare_rel.unpersist()

+--------+---------------+
|case_key|is_rare_variant|
+--------+---------------+
|C001    |false          |
|C002    |true           |
|C003    |true           |
|C004    |true           |
|C005    |true           |
|C006    |true           |
|C007    |false          |
+--------+---------------+

+--------+---------------+
|case_key|is_rare_variant|
+--------+---------------+
|C001    |false          |
|C002    |true           |
|C003    |true           |
|C004    |true           |
|C005    |true           |
|C006    |true           |
|C007    |false          |
+--------+---------------+



DataFrame[case_key: string, activity: string, timestamp: timestamp, resource: string, department: string, is_rare_variant: boolean]

In [10]:
# 4.8 variant_drift_over_time() — dense_rank per period; unordered by default
df_drift = variant_drift_over_time(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    time_window="month",
    include_rank_change=False,
)
show_df(
    df_drift.orderBy("period", "variant_rank", F.col("case_count").desc())
)

df_drift_chg = variant_drift_over_time(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    time_window="month",
    include_rank_change=True,
)
show_df(
    df_drift_chg.orderBy("period", "variant_rank", F.col("case_count").desc())
)

+-------------------+---------------------+----------+------------+
|period             |variant_fingerprint  |case_count|variant_rank|
+-------------------+---------------------+----------+------------+
|2026-03-01 00:00:00|A→B→C→D              |2         |1           |
|2026-03-01 00:00:00|A→C→D                |1         |2           |
|2026-03-01 00:00:00|A→B→A→C→D            |1         |2           |
|2026-03-01 00:00:00|A→B→P→C→D            |1         |2           |
|2026-03-01 00:00:00|A→B→Manual_Review→C→D|1         |2           |
|2026-03-01 00:00:00|A→B→B→C→D            |1         |2           |
+-------------------+---------------------+----------+------------+

+-------------------+---------------------+----------+------------+---------------------------+
|period             |variant_fingerprint  |case_count|variant_rank|rank_change_vs_prior_period|
+-------------------+---------------------+----------+------------+---------------------------+
|2026-03-01 00:00:00|A→B→C→D   

26/03/22 14:02:14 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1043648 ms exceeds timeout 120000 ms
26/03/22 14:02:14 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/22 14:02:14 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

## Validation checklist (Phase 4)

Re-run all cells after pulling changes; expect **no** `WindowExec: No Partition Defined` warnings from variant ranking (ranking is driver-side).

- [x] 4.1 `case_variant_fingerprint()`
- [x] 4.2 `variant_frequency_ranking()`
- [x] 4.3 `filter_top_n_variants()`
- [x] 4.4 `variant_coverage_ratio()`
- [x] 4.5 `variant_attribute_profile()`
- [x] 4.6 `detect_variant_loops()`
- [x] 4.7 `flag_rare_variants()`
- [x] 4.8 `variant_drift_over_time()`

Phase 4 lines in `PROGRESS.md` reflect this implementation (including March 2026 hardening).